# Project 2 — Goodreads Book Recommender

<small>EDA → collaborative filtering vs a popularity baseline (RMSE + Precision/Recall@N) → an **LLM re-ranking layer** that personalizes the Top-N to a stated preference. Reproducible from `data/Books.csv` and `data/Ratings.csv`. The logic lives in `src/`; this notebook is the narrative that becomes `app.py`.</small>


## Step 0 — Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # so `from src import ...` works from notebooks/
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import data_loader, cf_model, evaluate, recommend, llm_rerank
pd.set_option('display.max_colwidth', 55)


## Step 1 — Load the data
Reads the two assignment CSVs and repairs the author-name encoding.


In [ ]:
ratings, books = data_loader.load('../data')
print('ratings:', ratings.shape, '| books:', books.shape)
print('users:', ratings.user_id.nunique(), '| rated books:', ratings.book_id.nunique())
books[['book_id','title','authors','original_publication_year','average_rating','ratings_count']].head()


## Step 2 — Exploratory data analysis
Look at users, ratings, and books, and note what each implies for modeling.


In [ ]:
# Rating distribution — note the skew toward 4-5 stars (a 'missing not at random' hint)
ax = ratings['rating'].value_counts().sort_index().plot(kind='bar')
ax.set_title('Rating distribution'); ax.set_xlabel('stars'); ax.set_ylabel('count'); plt.show()
print(ratings['rating'].describe())


In [ ]:
# Sparsity + activity: ratings per user and per book
per_user = ratings.groupby('user_id').size()
per_book = ratings.groupby('book_id').size()
print(f'ratings/user  -> median {per_user.median():.0f}, max {per_user.max()}')
print(f'ratings/book  -> median {per_book.median():.0f}, max {per_book.max()}')
density = len(ratings) / (ratings.user_id.nunique() * ratings.book_id.nunique())
print(f'matrix density: {density:.4%}  (very sparse -> CF will be challenged)')


In [ ]:
# Popularity long tail: a few books soak up most ratings
per_book.sort_values(ascending=False).reset_index(drop=True).plot(
    title='Books ranked by #ratings (long tail)', logy=True)
plt.xlabel('book rank'); plt.ylabel('#ratings (log)'); plt.show()


In [ ]:
# Quality vs popularity, and the encoding fix sanity check
books.plot.scatter(x='ratings_count', y='average_rating', alpha=0.2, logx=True,
                   title='average_rating vs ratings_count'); plt.show()
print('language mix:'); print(books['language_code'].value_counts().head())
print('\nencoding repaired example:', books.loc[1, 'authors'])


### EDA takeaways
- Ratings skew high (4–5★) → popularity/mean is a **strong baseline**; CF must clear it.
- The user–item matrix is **very sparse** → expect CF to struggle on cold users/items.
- A heavy **popularity long tail** → watch popularity bias; track coverage.
- `average_rating` is compressed (~3.5–4.5) and weakly tied to popularity.
- Author encoding needed cleaning (handled in `load()`).


## Step 3 — Models: popularity baseline, UBCF, IBCF
All via `surprise` (UBCF Pearson, IBCF cosine) plus a non-personalized popularity/mean baseline. Held-out split for honest evaluation.


In [ ]:
train, test = evaluate.train_test_split_ratings(ratings, test_size=0.1)

pop = cf_model.PopularityModel().fit(train)
ubcf = cf_model.CFModel('ubcf').fit(train)
ibcf = cf_model.CFModel('ibcf').fit(train)

print('RMSE (lower is better):')
print('  Popularity baseline:', round(evaluate.rmse(pop,  test), 4))
print('  UBCF (Pearson)     :', round(evaluate.rmse(ubcf, test), 4))
print('  IBCF (cosine)      :', round(evaluate.rmse(ibcf, test), 4))


## Step 4 — Top-N ranking comparison (Precision@N / Recall@N)
RMSE measures rating accuracy; these measure the actual Top-N list quality. They don't always agree.


In [ ]:
def scorer(model):
    def f(user_id, book_ids):
        return model.predict_for_user(user_id, book_ids)
    return f

rows = []
for name, m in [('Popularity', pop), ('UBCF', ubcf), ('IBCF', ibcf)]:
    metrics = evaluate.ranking_metrics(scorer(m), train, test, k=10, threshold=4.0)
    metrics['RMSE'] = round(evaluate.rmse(m, test), 4)
    metrics['model'] = name
    rows.append(metrics)
pd.DataFrame(rows).set_index('model').round(4)


**Interpretation (write this up).** If the popularity baseline is hard to beat, that's the expected outcome on a sparse, popularity-skewed dataset — explain *why* (sparsity, cold items, head dominance) and what it implies for model selection. This honest analysis is worth points.


## Step 5 — Top-N from the best CF model, then the LLM layer
Pick the best model, generate a user's Top-N, then re-rank to a stated preference.


In [ ]:
best = ibcf   # set to whichever wins your comparison

class A:  # tiny adapter so recommend_top_n can use the raw model
    content = None
    def __init__(s, m): s.m = m
    def score(s, u, ur, ids): return s.m.predict_for_user(u, ids)

uid = ratings.user_id.value_counts().index[0]   # an active user
cf_recs = recommend.recommend_top_n(uid, A(best), ratings, books, top_n=10, min_ratings=50, explain=False)
cf_recs[['book_id','title','authors','score']]


In [ ]:
# LLM re-ranking: candidates -> Gemini (if GEMINI_API_KEY set) else heuristic fallback.
# Set your key in the environment first (never in code):  export GEMINI_API_KEY=...
cands = llm_rerank.candidates_from_recs(cf_recs, books)
picks, used_llm = llm_rerank.rerank(cands, preference='something dark and mysterious', top_k=5)
print('source:', 'Gemini LLM' if used_llm else 'heuristic fallback (no key)')
for i, p in enumerate(picks, 1):
    print(f'{i}. {p.title} — {p.authors}')
    print(f'   why: {p.explanation}')


## Step 6 — From notebook to Streamlit
This exact flow (pick user → CF Top-N → LLM re-rank) is wired in `../app.py`:

```bash
cd ..
export GEMINI_API_KEY=your_key_here   # optional; without it the app uses the fallback
streamlit run app.py
```

See `PROJECT_FRAMEWORK.md` for the rubric map and remaining milestones.
